# Tái dựng phòng hoặc vật thể bằng Nerfstudio

Notebook này xử lý dữ liệu bạn tự quay bằng COLMAP và train Splatfacto trên Colab T4. Mỗi cảnh có thư mục riêng trong Drive. Kết quả cuối là Gaussian PLY để xem bằng viewer của repo.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
from google.colab import drive

drive.mount("/content/drive")
CAPTURE_ID = "phong-01"
SOURCE_NAME = "capture.mp4"  # Hoặc capture.mov hay photos.zip
FRAMES = 100                 # Số frame mục tiêu nếu đầu vào là video
STEPS = 7000                 # Đặt 500 để thử nhanh
DOWNSCALE = 2                # 1, 2 hoặc 4
MATCHING = None              # Mặc định sequential cho video, exhaustive cho bộ ảnh

if not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_-]*", CAPTURE_ID):
    raise ValueError("CAPTURE_ID chỉ gồm chữ, số, dấu gạch ngang và gạch dưới.")
if Path(SOURCE_NAME).name != SOURCE_NAME:
    raise ValueError("SOURCE_NAME phải là tên file trong thư mục cảnh.")
if DOWNSCALE not in (1, 2, 4):
    raise ValueError("DOWNSCALE phải là 1, 2 hoặc 4.")
if FRAMES < 16 or STEPS < 1:
    raise ValueError("Cần ít nhất 16 frame mục tiêu và ít nhất một vòng train.")

PROJECT = Path("/content/drive/MyDrive/3dgs_captures") / CAPTURE_ID
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE = PROJECT / SOURCE_NAME
WORK = Path("/content/nerfstudio_lab")
WORK.mkdir(exist_ok=True)
ENV = WORK / "env"
OUTPUT = PROJECT / "runs"
OUTPUT.mkdir(exist_ok=True)
print("Đặt video hoặc photos.zip trong:", PROJECT)

def run(*args, env=None):
    command = [str(arg) for arg in args]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors="replace", env=env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


## 1. Cài môi trường GPU

Chọn T4 trong Colab. Lần đầu Splatfacto chạy, gsplat sẽ biên dịch các CUDA kernel.

In [ ]:
run("nvidia-smi")
mamba = WORK / "bin/micromamba"
if not mamba.exists():
    archive = WORK / "micromamba.tar.bz2"
    run("curl", "-fLsS", "-o", archive, "https://micro.mamba.pm/api/micromamba/linux-64/latest")
    run("tar", "-xjf", archive, "-C", WORK, "bin/micromamba")

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(WORK / "mamba")
env["MAX_JOBS"] = "2"
env["CUDA_HOME"] = str(ENV)
env["TORCH_CUDA_ARCH_LIST"] = "7.5"
# Avoid the Inductor driver-library lookup on Colab. gsplat still uses CUDA.
env["TORCHDYNAMO_DISABLE"] = "1"

def ns(*args):
    run(mamba, "run", "-p", ENV, *args, env=env)

marker = WORK / "setup-complete"
if not marker.exists():
    run(mamba, "create", "-y", "-p", ENV,
        "-c", "nvidia/label/cuda-11.8.0", "-c", "conda-forge",
        "python=3.10", "pip", "cuda-toolkit=11.8", "gxx_linux-64=11", env=env)
    ns("python", "-m", "pip", "install", "torch==2.1.2+cu118", "torchvision==0.16.2+cu118",
       "--index-url", "https://download.pytorch.org/whl/cu118")
    # PyTorch 2.1 imports pkg_resources when gsplat builds its extension.
    ns("python", "-m", "pip", "install", "setuptools==80.9.0",
       "numpy==1.26.4", "nerfstudio==1.1.5")
    ns("python", "-c", "import torch, gsplat; import torch.utils.cpp_extension; assert torch.cuda.is_available(); print(torch.__version__, gsplat.__version__)")
    ns("nvcc", "--version")
    marker.touch()

if not shutil.which("colmap") or not shutil.which("ffmpeg"):
    run("apt-get", "-qq", "update")
    run("apt-get", "-qq", "install", "-y", "colmap", "ffmpeg")


## 2. Xử lý video hoặc bộ ảnh của bạn

Video dùng ghép tuần tự theo mặc định. Bộ ảnh dùng ghép mọi cặp. Có thể đặt `MATCHING = "exhaustive"` nếu video quay vòng mà ghép tuần tự đăng ký được quá ít ảnh. Notebook lưu dataset đã xử lý lên Drive để tái sử dụng. Khi đổi nguồn hoặc cấu hình lấy mẫu, notebook tạo dataset mới.

In [ ]:
if not SOURCE.is_file():
    raise FileNotFoundError(f"Hãy tải file đầu vào lên Drive: {SOURCE}")

suffix = SOURCE.suffix.lower()
if suffix not in (".mp4", ".mov", ".zip"):
    raise ValueError("Đầu vào phải là video MP4, MOV hoặc ZIP chứa JPG/PNG.")
kind = "images" if suffix == ".zip" else "video"
matching = MATCHING or ("exhaustive" if kind == "images" else "sequential")
if matching not in ("sequential", "exhaustive"):
    raise ValueError("MATCHING phải là sequential, exhaustive hoặc None.")

with SOURCE.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()[:16]
dataset_id = f"v1-{kind}-{digest}-{matching}-{FRAMES if kind == 'video' else 'all'}"
DATA = WORK / CAPTURE_ID / dataset_id
archive = PROJECT / f"{dataset_id}.zip"
DATA.parent.mkdir(parents=True, exist_ok=True)

def validate_dataset(folder):
    metadata = json.loads((folder / "transforms.json").read_text(encoding="utf-8"))
    frames = metadata.get("frames", [])
    if len(frames) < 16:
        raise ValueError(f"Chỉ có {len(frames)} ảnh đăng ký. Hãy quay chậm và tăng độ chồng lấp.")
    seed = metadata.get("ply_file_path")
    if not seed or not (folder / seed).is_file():
        raise ValueError("COLMAP chưa tạo đám mây điểm thưa để khởi tạo Gaussian.")
    return len(frames)

if not DATA.is_dir():
    with tempfile.TemporaryDirectory(dir=DATA.parent) as temporary:
        temp = Path(temporary)
        result = temp / "processed"
        if archive.is_file():
            print("Đang khôi phục dataset:", archive.name, flush=True)
            result.mkdir()
            shutil.unpack_archive(str(archive), str(result))
        else:
            if kind == "images":
                unpacked = temp / "unpacked"
                shutil.unpack_archive(str(SOURCE), str(unpacked))
                photos = sorted(
                    p for p in unpacked.rglob("*")
                    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
                    and "__MACOSX" not in p.relative_to(unpacked).parts
                )
                if len(photos) < 16:
                    raise ValueError("ZIP cần ít nhất 16 ảnh JPG/PNG.")
                input_path = temp / "images"
                input_path.mkdir()
                for index, photo in enumerate(photos):
                    shutil.copyfile(photo, input_path / f"{index:05d}{photo.suffix.lower()}")
            else:
                input_path = temp / f"capture{suffix}"
                shutil.copyfile(SOURCE, input_path)
            flags = ["--num-frames-target", FRAMES] if kind == "video" else []
            ns("ns-process-data", kind, "--data", input_path, "--output-dir", result,
               "--sfm-tool", "colmap", "--matching-method", matching, "--no-gpu",
               "--num-downscales", 2, "--verbose", *flags)
            validate_dataset(result)
            local_zip = Path(shutil.make_archive(str(temp / "dataset"), "zip", result))
            pending = archive.with_suffix(".zip.part")
            shutil.copyfile(local_zip, pending)
            pending.replace(archive)
        validate_dataset(result)
        result.rename(DATA)

print(f"COLMAP đăng ký {validate_dataset(DATA)} ảnh. Dataset: {DATA}")


## 3. Train Splatfacto

Để kiểm tra cài đặt và camera trước, có thể đặt `STEPS = 500` trong một ô riêng rồi chạy mục này. Sau đó chạy lại với 7.000 vòng để tạo baseline chất lượng hơn.

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
experiment = f"splatfacto-{STEPS}"
ns("ns-train", "splatfacto",
   "--data", DATA, "--output-dir", OUTPUT, "--experiment-name", experiment,
   "--timestamp", stamp, "--max-num-iterations", STEPS,
   "--steps-per-save", min(1000, STEPS), "--vis", "tensorboard",
   "--pipeline.datamanager.cache-images", "cpu",
   "nerfstudio-data", "--eval-mode", "interval", "--eval-interval", 8,
   "--downscale-factor", DOWNSCALE, "--load-3D-points", "True")
CONFIG = OUTPUT / experiment / "splatfacto" / stamp / "config.yml"
assert CONFIG.is_file(), f"Không tìm thấy cấu hình: {CONFIG}"
print("Cấu hình:", CONFIG)


## 4. Đánh giá và xuất PLY

Tải `splat.ply` từ đường dẫn được in rồi mở trong **Gaussian Lab Viewer**.

In [ ]:
export = CONFIG.parent / "export"
export.mkdir(exist_ok=True)
ns("ns-eval", "--load-config", CONFIG, "--output-path", export / "metrics.json")
ns("ns-export", "gaussian-splat", "--load-config", CONFIG, "--output-dir", export)
ply = export / "splat.ply"
assert ply.is_file(), "Chưa tạo được splat.ply."
print("Tải PLY từ Drive:", ply)
